# FY2026 Committee of Supply: How MPs Spent Their Speaking Time

## What is the Committee of Supply?

The Committee of Supply (COS) is the annual budget debate in Singapore's Parliament. After the Finance Minister delivers the Budget Statement, Parliament resolves into a Committee of the whole House to examine each ministry's spending estimates.

**How it works (Standing Order 92):**

- The Speaker allots **7 days** for COS debates (can be extended)
- Each ministry ("head of expenditure") gets a fixed time block. The Speaker sets **"guillotine" deadlines** — when time's up, debate stops and the budget is voted on
- MPs participate by filing **"cuts"** — formal amendments to reduce a ministry's budget by $100. This is a procedural device; the point isn't to cut funding, but to raise issues on the record
- **PAP backbenchers get 18 minutes** per cut. **Opposition MPs and GPC chairs get 20 minutes**
- After the main speeches, MPs can ask **follow-up clarifications** during a clarification segment
- Ministers and their teams respond to the cuts and clarifications

**Why this matters:** COS is the one time a year when every backbencher gets to put questions to ministers on the record. What they choose to speak about — and how they allocate their limited time across ministries — reveals their policy priorities.

## Methodology

We use **word count as a proxy for speaking time**, since the Hansard doesn't include per-speech timestamps. This is imperfect — a fast speaker covers more words per minute — but it's the best available measure of how much airtime each MP claimed. Word counts include both main speeches and follow-up clarifications.

Data scraped from the Singapore Parliament Hansard API using `sgparl`.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Load data
speeches = pd.read_csv('../data/budget2026/speeches.csv')
topics = pd.read_csv('../data/budget2026/topics.csv')
sittings = pd.read_csv('../data/budget2026/sittings.csv')

print(f"Budget period: {speeches.date.min()} to {speeches.date.max()}")
print(f"Total speeches: {len(speeches):,}")
print(f"Total topics: {len(topics):,}")
print(f"Sitting days: {len(sittings)}")

## 1. Extract COS speeches by ministry

### EDITORIAL CHOICES in this cell

> **[SCOPE]** We only analyse speeches from topics titled "Committee of Supply – Head X (Ministry of Y)". This excludes the Budget Statement debate itself (24-25 Feb), oral questions during budget week, and written answers. The COS is where backbenchers directly engage ministers on specific policy areas — that's the signal we want.

> **[EXCLUSION]** We exclude topics titled "Reporting Progress", "Time Allocation", "Commencement Time", "Total Sums", and "Heads B, C, D, E, G and Z" (omnibus vote). These are procedural — no substantive MP speeches.

> **[EXCLUSION]** We exclude rows where `member_name` is "Speaker", "Deputy Speaker", or "Speaker in the Chair" — these are the Chair calling on MPs, not substantive speeches. We also exclude rows with empty `member_name` — these are "The Chairman" procedural entries during COS debates.

> **[CLASSIFICATION]** We identify officeholders (ministers, ministers of state, senior parliamentary secretaries) by checking if `member_name_original` contains "Minister" or "Parliamentary Secretary" in **any** of their COS speeches. If an MP ever appears with a ministerial title, **all** their COS speeches are classified as ministerial. This is the conservative choice — it means we may undercount backbencher participation for dual-role MPs (e.g. a Minister of State who also files cuts on another ministry), but it avoids inflating the backbencher numbers with ministerial answers.

In [ ]:
# Filter to COS topics and extract ministry names
cos_topics = topics[topics.title.str.contains('Committee of Supply', case=False, na=False)].copy()
cos_topics = cos_topics[~cos_topics.title.str.contains(
    'Reporting Progress|Time Allocation|Commencement|Total Sums|Heads B',
    case=False, na=False
)]

def extract_ministry(title):
    m = re.search(r'\(([^)]+)\)', title)
    return m.group(1).replace('Ministry of ', '') if m else None

cos_topics['ministry'] = cos_topics['title'].apply(extract_ministry)
cos_topics = cos_topics[cos_topics.ministry.notna()]

# Merge speeches with ministry
cos = speeches.merge(cos_topics[['topic_id', 'ministry']], on='topic_id')

# Drop procedural speakers
cos = cos[
    cos.member_name.notna() &
    (cos.member_name != '') &
    ~cos.member_name.isin(['Speaker', 'Deputy Speaker', 'Speaker in the Chair'])
]

# Identify officeholders: anyone who EVER appears with a ministerial title
# This catches ministers, ministers of state, and senior parliamentary secretaries
officeholder_names = cos[
    cos.member_name_original.str.contains('Minister|Parliamentary Secretary', case=False, na=False)
].member_name.unique()

cos['is_officeholder'] = cos.member_name.isin(officeholder_names)

backbenchers = cos[~cos.is_officeholder].copy()
officeholders = cos[cos.is_officeholder].copy()

print(f"COS ministries debated: {cos.ministry.nunique()}")
print(f"COS date range: {cos.date.min()} to {cos.date.max()}")
print(f"Total COS speeches: {len(cos):,}")
print(f"  Officeholder speeches: {len(officeholders):,} ({officeholders.num_words.sum():,} words) — {len(officeholder_names)} officeholders")
print(f"  Backbencher speeches: {len(backbenchers):,} ({backbenchers.num_words.sum():,} words) — {backbenchers.member_name.nunique()} backbenchers")
print(f"\nOfficeholders identified:")
for name in sorted(officeholder_names):
    print(f"  {name}")

## 2. How each MP allocated their COS time across ministries

### EDITORIAL CHOICES in this cell

> **[ASSUMPTION]** Word count is used as a proxy for speaking time. A typical parliamentary speaking pace is ~120-150 words per minute, so 1,000 words ≈ 7-8 minutes. This is imperfect — some MPs speak faster than others — but it's the best available measure from the Hansard text.

> **[SCOPE]** This table shows **backbenchers only** (not ministers). Ministers speak because they have to respond; backbenchers speak because they chose to. That choice is the editorial signal.

> **[INCLUSION]** Word counts include both the MP's **main COS speech** (typically 18 minutes for PAP backbenchers, 20 minutes for opposition MPs and GPC chairs) **and any follow-up clarifications** they make during the clarification segment. The Hansard does not distinguish between the two — both are recorded as speech paragraphs under the same COS topic. This means the word counts reflect **total airtime used**, not just the prepared speech. An MP who asked aggressive follow-ups would show higher word counts than one who delivered their speech and sat down.

In [ ]:
# Each backbencher's word allocation across ministries
bb_by_ministry = backbenchers.groupby(['member_name', 'party', 'ministry']).agg(
    speeches=('speech_id', 'count'),
    words=('num_words', 'sum')
).reset_index()

# Total words per MP
bb_totals = bb_by_ministry.groupby(['member_name', 'party']).words.sum().reset_index(name='total_words')
bb_totals = bb_totals.sort_values('total_words', ascending=False)

# Show top 30 backbenchers with their ministry breakdown
print("Top 30 backbenchers by COS word count (≈ speaking time)")
print("=" * 80)
for _, mp in bb_totals.head(30).iterrows():
    interests = bb_by_ministry[bb_by_ministry.member_name == mp.member_name].sort_values('words', ascending=False)
    ministry_str = ' | '.join([f"{r.ministry} ({r.words:,}w)" for _, r in interests.iterrows()])
    est_minutes = round(mp.total_words / 135)  # ~135 wpm average
    print(f"\n{mp.member_name} ({mp.party}) — {mp.total_words:,} words (~{est_minutes} min)")
    print(f"  {ministry_str}")

## 3. Heatmap: MP × Ministry word allocation

### EDITORIAL CHOICES in this cell

> **[VISUAL]** We show the top 40 backbenchers (by total COS words) on the y-axis, and all ministries on the x-axis. MPs are sorted by total words (most active at top). Colour intensity represents word count per cell. We use a log-normalised colour scale to avoid a few high-volume cells washing out the rest.

> **[THRESHOLD]** Top 40 is chosen to keep the chart readable. This captures ~85% of backbencher COS activity. The remaining ~60 MPs spoke very little (under 500 words total — roughly one 4-minute speech).

In [ ]:
# Build pivot table: MP x Ministry
top_n = 40
top_mps = bb_totals.head(top_n).member_name.tolist()

pivot = bb_by_ministry[bb_by_ministry.member_name.isin(top_mps)].pivot_table(
    index='member_name', columns='ministry', values='words', fill_value=0
)

# Reorder rows by total words
pivot = pivot.loc[top_mps]

# Shorten ministry names for display
short_names = {
    'Culture, Community and Youth': 'MCCY',
    'Defence': 'MINDEF',
    'Digital Development and Information': 'MDDI',
    'Education': 'MOE',
    'Finance': 'MOF',
    'Foreign Affairs': 'MFA',
    'Health': 'MOH',
    'Home Affairs': 'MHA',
    'Law': 'MinLaw',
    'Manpower': 'MOM',
    'National Development': 'MND',
    'Social and Family Development': 'MSF',
    'Sustainability and the Environment': 'MSE',
    'Trade and Industry': 'MTI',
    'Transport': 'MOT',
    'Parliament': 'Parl',
    'Prime Minister\'s Office': 'PMO',
}
pivot.columns = [short_names.get(c, c) for c in pivot.columns]

# Add party labels to MP names
party_map = dict(zip(bb_totals.member_name, bb_totals.party))
pivot.index = [f"{name} ({party_map[name]})" for name in pivot.index]

# Plot
fig, ax = plt.subplots(figsize=(16, 14))

# Log-normalised colours (add 1 to avoid log(0))
from matplotlib.colors import LogNorm
data = pivot.values.astype(float)
data_plot = np.where(data > 0, data, np.nan)

im = ax.imshow(data_plot, cmap='YlOrRd', aspect='auto',
               norm=LogNorm(vmin=1, vmax=data.max()))

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)

# Add word count text in cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = int(data[i, j])
        if val > 0:
            color = 'white' if val > 2000 else 'black'
            ax.text(j, i, f'{val:,}', ha='center', va='center', fontsize=6, color=color)

plt.colorbar(im, ax=ax, label='Words', shrink=0.6)
ax.set_title('FY2026 Committee of Supply: Backbencher Word Allocation by Ministry', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

## 4. PAP backbenchers vs WP vs NMP: participation patterns

### EDITORIAL CHOICES in this cell

> **[COMPARISON]** We compare three groups: PAP backbenchers (79 MPs), WP (12 MPs including 2 NCMPs), and NMPs (10). The comparison is per-MP (average words per MP) not total — since PAP has 6× more MPs, raw totals would be misleading.

> **[CLASSIFICATION]** "PAP backbencher" means PAP MPs who did not speak in a ministerial capacity during COS. Some PAP MPs hold office-holder roles (Minister of State, Senior Parliamentary Secretary) but also filed cuts on other ministries in a personal capacity. These dual-role speeches are hard to separate, so we classify based on whether the `member_name_original` field contained a ministerial title for that specific speech.

In [ ]:
# Party comparison
party_stats = backbenchers.groupby('party').agg(
    unique_mps=('member_name', 'nunique'),
    total_speeches=('speech_id', 'count'),
    total_words=('num_words', 'sum'),
    ministries_covered=('ministry', 'nunique')
).reset_index()

party_stats['words_per_mp'] = (party_stats.total_words / party_stats.unique_mps).round(0).astype(int)
party_stats['speeches_per_mp'] = (party_stats.total_speeches / party_stats.unique_mps).round(1)
party_stats['ministries_per_mp'] = (
    backbenchers.groupby(['party', 'member_name']).ministry.nunique()
    .reset_index().groupby('party').ministry.mean()
).round(1).values

print("COS participation by party (backbenchers only)")
print("=" * 75)
print(party_stats.to_string(index=False))

# Visualise words per MP by party
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: words per MP
colors = {'PAP': '#1e88e5', 'WP': '#e53935', 'NMP': '#43a047'}
ax = axes[0]
for i, (_, row) in enumerate(party_stats.iterrows()):
    ax.bar(row.party, row.words_per_mp, color=colors.get(row.party, 'gray'))
    ax.text(i, row.words_per_mp + 50, f"{row.words_per_mp:,}", ha='center', fontsize=10)
ax.set_ylabel('Average words per MP')
ax.set_title('COS speaking volume per MP by party')

# Bar chart: ministries per MP
ax = axes[1]
for i, (_, row) in enumerate(party_stats.iterrows()):
    ax.bar(row.party, row.ministries_per_mp, color=colors.get(row.party, 'gray'))
    ax.text(i, row.ministries_per_mp + 0.05, f"{row.ministries_per_mp}", ha='center', fontsize=10)
ax.set_ylabel('Average ministries spoken on per MP')
ax.set_title('COS breadth of engagement by party')

plt.tight_layout()
plt.show()

## 5. Focus vs breadth: which MPs spread thin, which went deep?

### EDITORIAL CHOICES in this cell

> **[CLASSIFICATION]** We define "focused" as MPs who put 70%+ of their COS words into a single ministry, and "broad" as MPs who spoke on 5+ ministries. These thresholds are judgment calls — an MP speaking on 4 ministries could reasonably be called "broad" too. The 70% threshold was chosen because it means more than two-thirds of their time went to one area, a clear concentration.

> **[THRESHOLD] (UNCERTAIN)** We only include MPs with 500+ total COS words (~4 minutes of speaking time). Below that, the "focus" metric is unreliable — one short question on one ministry looks like 100% focus but means very little.

In [ ]:
# Calculate focus metrics for each backbencher
mp_focus = []
for name, group in bb_by_ministry.groupby('member_name'):
    total = group.words.sum()
    if total < 500:
        continue
    top_ministry = group.sort_values('words', ascending=False).iloc[0]
    mp_focus.append({
        'member_name': name,
        'party': group.party.iloc[0],
        'total_words': total,
        'num_ministries': len(group),
        'top_ministry': top_ministry.ministry,
        'top_ministry_words': top_ministry.words,
        'top_ministry_pct': round(top_ministry.words / total * 100, 1),
    })

focus_df = pd.DataFrame(mp_focus).sort_values('top_ministry_pct', ascending=False)

# Focused MPs (70%+ on one ministry)
focused = focus_df[focus_df.top_ministry_pct >= 70].sort_values('top_ministry_pct', ascending=False)
print(f"FOCUSED MPs (70%+ of COS words on one ministry): {len(focused)}")
print("=" * 80)
for _, r in focused.iterrows():
    print(f"  {r.member_name:30s} ({r.party:3s}) {r.top_ministry_pct:5.1f}% on {r.top_ministry} ({r.total_words:,}w total)")

# Broad MPs (5+ ministries)
broad = focus_df[focus_df.num_ministries >= 5].sort_values('num_ministries', ascending=False)
print(f"\nBROAD MPs (spoke on 5+ ministries): {len(broad)}")
print("=" * 80)
for _, r in broad.iterrows():
    print(f"  {r.member_name:30s} ({r.party:3s}) {r.num_ministries} ministries ({r.total_words:,}w total)")

# Scatter plot: num ministries vs total words
fig, ax = plt.subplots(figsize=(12, 7))
for party, color in colors.items():
    subset = focus_df[focus_df.party == party]
    ax.scatter(subset.num_ministries, subset.total_words, c=color, label=party,
               s=60, alpha=0.7, edgecolors='white', linewidths=0.5)
    for _, r in subset.iterrows():
        if r.total_words > 4000 or r.num_ministries >= 6:
            ax.annotate(r.member_name, (r.num_ministries, r.total_words),
                       fontsize=7, ha='left', va='bottom',
                       xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Number of ministries spoken on')
ax.set_ylabel('Total COS words')
ax.set_title('COS engagement: depth vs breadth (backbenchers with 500+ words)')
ax.legend()
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 6. Which ministries attracted the most backbencher attention?

### EDITORIAL CHOICES in this cell

> **[COMPARISON]** We compare ministries by total backbencher words — not minister words. Minister word counts reflect how much they had to answer, not what MPs cared about. A ministry with lots of backbencher words means lots of MPs chose to spend their limited time there.

> **[VISUAL]** Stacked bar by party shows whether attention came from PAP backbenchers, WP, or NMPs. This helps distinguish between "PAP MPs asking friendly questions" and "opposition scrutiny".

In [ ]:
# Ministry attention by party
ministry_party = backbenchers.groupby(['ministry', 'party']).agg(
    mps=('member_name', 'nunique'),
    words=('num_words', 'sum')
).reset_index()

# Pivot for stacked bar
ministry_pivot = ministry_party.pivot_table(
    index='ministry', columns='party', values='words', fill_value=0
)
ministry_pivot['total'] = ministry_pivot.sum(axis=1)
ministry_pivot = ministry_pivot.sort_values('total', ascending=True)

# Shorten names
ministry_pivot.index = [short_names.get(m, m) for m in ministry_pivot.index]

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

parties_ordered = ['PAP', 'WP', 'NMP']
bottom = np.zeros(len(ministry_pivot))
for party in parties_ordered:
    if party in ministry_pivot.columns:
        vals = ministry_pivot[party].values
        ax.barh(ministry_pivot.index, vals, left=bottom,
                color=colors.get(party, 'gray'), label=party)
        bottom += vals

ax.set_xlabel('Total backbencher words')
ax.set_title('COS FY2026: Backbencher attention by ministry')
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

# Also show unique MP counts per ministry
print("\nUnique backbenchers per ministry:")
print("=" * 60)
mp_counts = backbenchers.groupby('ministry').member_name.nunique().sort_values(ascending=False)
for ministry, count in mp_counts.items():
    total_words = backbenchers[backbenchers.ministry == ministry].num_words.sum()
    print(f"  {short_names.get(ministry, ministry):8s}  {count:3d} MPs  {total_words:>7,} words")

## Methodological Appendix: All Editorial Choices

| # | Tag | Choice | Alternative | Consequence | Confidence |
|---|-----|--------|-------------|-------------|------------|
| 1 | SCOPE | COS topics only (not Budget Statement debate, oral questions, or written answers) | Include all budget-period speeches | Would blur the signal — COS is where backbenchers choose their topics | High |
| 2 | EXCLUSION | Drop "Reporting Progress", "Time Allocation", "Commencement", "Total Sums", "Heads B,C,D,E,G,Z" topics | Include them | These are procedural — no substantive MP speeches | High |
| 3 | EXCLUSION | Drop Speaker/Deputy Speaker/Chairman rows | Include them | These are procedural calls, not substantive speeches | High |
| 4 | CLASSIFICATION | Officeholders identified by "Minister" or "Parliamentary Secretary" in `member_name_original` across ANY of their COS speeches | Per-speech classification | Conservative — may exclude dual-role MPs who also filed personal cuts on other ministries | High |
| 5 | ASSUMPTION | Word count ≈ speaking time (~135 wpm) | Actual timestamps (not available) | Fast speakers underrepresented, slow speakers overrepresented | Medium |
| 6 | INCLUSION | Word counts include both main COS speeches (18 min PAP / 20 min opposition+GPC chairs) AND follow-up clarifications | Separate them | Hansard doesn't distinguish — both recorded as paragraphs under same topic. Reflects total airtime used | High |
| 7 | THRESHOLD | Top 40 MPs in heatmap | Show all ~68 | Bottom ~28 have <500 words total — noise, not signal | High |
| 8 | VISUAL | Log-normalised heatmap colours | Linear scale | Linear makes a few 5,000-word cells wash out everything else | High |
| 9 | COMPARISON | Per-MP averages for party comparison (not totals) | Raw totals | PAP has 6× more MPs — totals would be misleading | High |
| 10 | CLASSIFICATION | "Focused" = 70%+ of words on one ministry | 60% or 80% | 70% = more than two-thirds, a clear concentration | Medium |
| 11 | THRESHOLD | Only include MPs with 500+ total COS words in focus analysis | Include all | One short question looks like 100% focus — unreliable below 500w | High |
| 12 | LIMITATION | Colons stripped from text by upstream parser | Fix parser | Ratios like "1:20" appear as "1 20" — doesn't affect word counts | N/A |
| 13 | LIMITATION | Some dual-role MPs (e.g. Ministers of State) speak both as officeholders and backbenchers across different ministries — hard to fully separate | Manual review of each speech | May exclude some legitimate backbencher cuts from officeholders | Low |